# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [2]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [3]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer
License(s): ODbL-1.0


100%|█████████████████████████████████████| 32.7M/32.7M [00:03<00:00, 11.4MB/s]



Connected to database at: /var/folders/p9/vd2t0bjd711gt4c08zmm0w4c0000gp/T/database.sqlite


---

## Loading SQL Data

In [4]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [20]:
# Load any table into a DataFrame
df = pd.read_sql("""
                 
                SELECT *
                FROM Match
                WHERE league_id = 13274 AND season = '2009/2010'
                
                    """, connection)
display(df)

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,13580,13274,13274,2009/2010,1,2009-08-01 00:00:00,662248,8525,8277,3,...,2.50,2.62,3.20,2.38,2.50,3.25,2.60,2.60,3.2,2.50
1,13581,13274,13274,2009/2010,1,2009-08-01 00:00:00,662249,10219,9908,0,...,2.25,2.60,3.25,2.40,2.90,3.25,2.30,2.75,3.2,2.38
2,13582,13274,13274,2009/2010,1,2009-08-01 00:00:00,662250,8614,8611,0,...,1.62,5.00,3.50,1.62,5.00,3.50,1.65,5.00,3.5,1.62
3,13583,13274,13274,2009/2010,1,2009-08-01 00:00:00,662251,9791,10229,3,...,1.44,6.00,3.60,1.50,6.50,3.75,1.50,6.50,3.6,1.50
4,13584,13274,13274,2009/2010,1,2009-08-02 00:00:00,662252,8640,9839,3,...,13.00,1.11,6.50,17.00,1.15,7.00,13.00,1.14,6.5,15.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,13881,13274,13274,2009/2010,9,2009-10-03 00:00:00,662323,10217,8674,1,...,2.60,2.70,3.30,2.30,2.50,3.25,2.55,2.50,3.3,2.50
302,13882,13274,13274,2009/2010,9,2009-10-04 00:00:00,662324,10228,9839,1,...,6.50,1.57,3.25,6.00,1.60,3.50,5.25,1.57,3.5,5.50
303,13883,13274,13274,2009/2010,9,2009-10-04 00:00:00,662325,9803,8593,2,...,1.40,6.50,3.75,1.44,6.25,4.00,1.45,6.50,3.6,1.50
304,13884,13274,13274,2009/2010,9,2009-10-04 00:00:00,662326,9908,8640,0,...,1.80,4.33,3.30,1.75,4.00,3.40,1.80,4.00,3.3,1.83
